# Hinge loss SVM

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDClassifier

In [ ]:
DATA_PATH = r"D:\墨大sml作业\FeatureA"

feature_path = DATA_PATH + r"\feature_A.csv"

df = pd.read_csv(feature_path)

drop_cols = ["userId", "movieId", "label"]

X_all = df.drop(columns=drop_cols).values
y_all = df["label"].values

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)

X shape: (20000263, 14)
y shape: (20000263,)


In [3]:
def stratified_split_indices(y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)

    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]

    rng.shuffle(idx_0)
    rng.shuffle(idx_1)

    n_test_0 = int(len(idx_0) * test_size)
    n_test_1 = int(len(idx_1) * test_size)

    test_idx = np.concatenate([idx_0[:n_test_0], idx_1[:n_test_1]])
    train_idx = np.concatenate([idx_0[n_test_0:], idx_1[n_test_1:]])

    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return train_idx, test_idx

In [4]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

In [5]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)

    pos_ranks = ranks[y_true == 1]
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)

    if n_pos == 0 or n_neg == 0:
        return np.nan

    auc = (np.sum(pos_ranks) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auc

In [6]:
def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (TP + TN) / len(y_true)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    }

In [7]:
def tune_svm_alpha_from_scratch(X_train, y_train, alpha_values, inner_repeats=3, val_size=0.2, seed=100):
    alpha_scores = {}

    for alpha in alpha_values:
        fold_scores = []

        for r in range(inner_repeats):
            inner_train_idx, val_idx = stratified_split_indices(
                y_train,
                test_size=val_size,
                seed=seed + r
            )

            X_inner_train = X_train[inner_train_idx]
            y_inner_train = y_train[inner_train_idx]

            X_val = X_train[val_idx]
            y_val = y_train[val_idx]

            X_inner_train_scaled, X_val_scaled = standardize_train_test(
                X_inner_train,
                X_val
            )

            svm_model = SGDClassifier(
                loss="hinge",
                alpha=alpha,
                max_iter=1000,
                tol=1e-3,
                random_state=42
            )

            svm_model.fit(X_inner_train_scaled, y_inner_train)

            y_val_pred = svm_model.predict(X_val_scaled)
            y_val_score = svm_model.decision_function(X_val_scaled)

            metrics = compute_metrics_from_scratch(
                y_val,
                y_val_pred,
                y_val_score
            )

            fold_scores.append(metrics["f1"])

        alpha_scores[alpha] = np.mean(fold_scores)

    best_alpha = max(alpha_scores, key=alpha_scores.get)

    return best_alpha, alpha_scores

In [8]:
alpha_values = [0.0001, 0.001, 0.01]

outer_results = []

for repeat in range(10):
    print(f"\nOuter repeat {repeat + 1}/10")

    train_idx, test_idx = stratified_split_indices(
        y_all,
        test_size=0.2,
        seed=42 + repeat
    )

    X_train = X_all[train_idx]
    y_train = y_all[train_idx]

    X_test = X_all[test_idx]
    y_test = y_all[test_idx]

    best_alpha, alpha_scores = tune_svm_alpha_from_scratch(
        X_train,
        y_train,
        alpha_values=alpha_values,
        inner_repeats=3,
        val_size=0.2,
        seed=2000 + repeat * 10
    )

    print("Alpha scores:", alpha_scores)
    print("Best alpha:", best_alpha)

    X_train_scaled, X_test_scaled = standardize_train_test(
        X_train,
        X_test
    )

    final_svm = SGDClassifier(
        loss="hinge",
        alpha=best_alpha,
        max_iter=1000,
        tol=1e-3,
        random_state=42
    )

    final_svm.fit(X_train_scaled, y_train)

    y_pred = final_svm.predict(X_test_scaled)
    y_score = final_svm.decision_function(X_test_scaled)

    metrics = compute_metrics_from_scratch(
        y_test,
        y_pred,
        y_score
    )

    metrics["repeat"] = repeat + 1
    metrics["best_alpha"] = best_alpha

    outer_results.append(metrics)


Outer repeat 1/10
Alpha scores: {0.0001: np.float64(0.723430379996234), 0.001: np.float64(0.7233873819829362), 0.01: np.float64(0.7240153474832889)}
Best alpha: 0.01

Outer repeat 2/10
Alpha scores: {0.0001: np.float64(0.7231051230742406), 0.001: np.float64(0.7229008506794532), 0.01: np.float64(0.7237616113328404)}
Best alpha: 0.01

Outer repeat 3/10
Alpha scores: {0.0001: np.float64(0.7226522146948344), 0.001: np.float64(0.7234779246331112), 0.01: np.float64(0.7238146068603907)}
Best alpha: 0.01

Outer repeat 4/10
Alpha scores: {0.0001: np.float64(0.7240005425254415), 0.001: np.float64(0.7233593003623685), 0.01: np.float64(0.7238455693771604)}
Best alpha: 0.0001

Outer repeat 5/10
Alpha scores: {0.0001: np.float64(0.7214770229036063), 0.001: np.float64(0.7230009902906352), 0.01: np.float64(0.7240156437806785)}
Best alpha: 0.01

Outer repeat 6/10
Alpha scores: {0.0001: np.float64(0.7233851129365433), 0.001: np.float64(0.7233688600730552), 0.01: np.float64(0.7239490823782643)}
Best alp

In [9]:
svm_results_df = pd.DataFrame(outer_results)

svm_results_df

,accuracy,precision,recall,f1,auc,TP,TN,FP,FN,repeat,best_alpha
0,0.719332,0.711724,0.736852,0.724070,0.794375,1473027,1404337,596633,526055,1,0.0100
1,0.719372,0.711863,0.736648,0.724044,0.794502,1472620,1404906,596064,526462,2,0.0100
2,0.719668,0.712082,0.737109,0.724379,0.794773,1473541,1405169,595801,525541,3,0.0100
3,0.719138,0.712874,0.733407,0.722994,0.794304,1466140,1410448,590522,532942,4,0.0001
4,0.719040,0.711434,0.736579,0.723788,0.794245,1472482,1403715,597255,526600,5,0.0100
5,0.719456,0.712495,0.735393,0.723763,0.794520,1470111,1407752,593218,528971,6,0.0100
6,0.719115,0.711632,0.736350,0.723780,0.794299,1472025,1404474,596496,527057,7,0.0100
7,0.719418,0.711809,0.736935,0.724154,0.794586,1473193,1404517,596453,525889,8,0.0100
8,0.719105,0.711704,0.736140,0.723716,0.794396,1471604,1404855,596115,527478,9,0.0100
9,0.719487,0.711924,0.736884,0.724189,0.794358,1473091,1404893,596077,525991,10,0.0100


In [10]:
metric_cols = ["accuracy", "precision", "recall", "f1", "auc"]

svm_summary_df = pd.DataFrame({
    "mean": svm_results_df[metric_cols].mean(),
    "std": svm_results_df[metric_cols].std()
})

svm_summary_df

,mean,std
accuracy,0.719313,0.000205
precision,0.711954,0.000431
recall,0.736230,0.001109
f1,0.723888,0.000384
auc,0.794436,0.000160


In [11]:
results_path = DATA_PATH + r"\SVM_FeatureA_repeated_results.csv"
summary_path = DATA_PATH + r"\SVM_FeatureA_summary.csv"

svm_results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
svm_summary_df.to_csv(summary_path, encoding="utf-8-sig")

print("Saved:")
print(results_path)
print(summary_path)

Saved:
D:\墨大sml作业\Feature1\SVM_FeatureA_repeated_results.csv
D:\墨大sml作业\Feature1\SVM_FeatureA_summary.csv
